# Run independent folds with an explicit CPU budget

## Goal and setup

One fold is one job. Two small synthetic folds fit independent pipelines in local
processes; results retain fold order and match serial predictions. The worker PID
table shows where each fit ran. Use the existing Python (misc314) kernel. No real
football export is loaded and no speed or forecasting claim is made.

### 1. Compare serial and parallel predictions

execution=None preserves the earlier serial behavior. An explicit policy also
sets the per-fit thread budget. Processes can be reused across folds.

In [1]:
from pathlib import Path
import os, sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
source_root = Path(os.environ.get("XDIYO_VERIFIED_SOURCE", str(root)))
sys.path[:0] = [str(source_root / "src"), str(source_root), str(root)]
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from notebooks.helpers.execution_demo import prepared_demo
from xdiyo_analytics.training import ExecutionPolicy, EstimatorAdapter, TrainingRunner

output = root / "experiment/execution_policy_demo/notebook_revised"
data, splits = prepared_demo()

def model_factory():
    return EstimatorAdapter(make_pipeline(StandardScaler(), Ridge(alpha=.1)))

serial = TrainingRunner(model_factory).run(data, splits)
policy = ExecutionPolicy(n_jobs=2, device="cpu", inner_threads=1)
parallel = TrainingRunner(model_factory, execution=policy).run(data, splits)
assert [fold.fold_id for fold in parallel.folds] == [0, 1]
for expected, actual in zip(serial.folds, parallel.folds):
    pd.testing.assert_frame_equal(expected.predictions["predict"], actual.predictions["predict"])
    assert actual.training_summary["execution"]["worker_pid"] != os.getpid()
execution_table = pd.DataFrame([
    {"fold": fold.fold_id, **fold.training_summary["execution"]} for fold in parallel.folds
])
print(execution_table)


   fold requested_device device  worker_pid  inner_threads
0     0              cpu    cpu       67344              1
1     1              cpu    cpu       35620              1


### 2. Reuse a completed experiment and inspect its predictions

The same policy reaches experiment fitting. The additional deployment refit below
is explicitly requested on all sixteen synthetic rows and runs in this process;
the held-out report still uses the fold evaluation results. Repeated execution
reopens saved numerical results with model=None. Choose a new output directory to
run a separate demonstration.

Candidates and outer nested-CV loops stay sequential. In parallel fits, observer
events appear in the parent when each fold completes; serial observers stay live.

In [2]:
from xdiyo_analytics.experiments import FootballExperiment, PreparedExperiment, RefitPolicy
from xdiyo_analytics.selection import Candidate
from xdiyo_analytics.analysis import PostTrainingAnalysis
from xdiyo_analytics.reporting import PerformanceReporter, MatchResultReporter

experiment = FootballExperiment("Synthetic fold execution", output_dir=output)
prepared = PreparedExperiment(data, splits, config={"synthetic": True})
candidate = Candidate("Ridge", model_factory, config={"alpha": .1})
reports = PostTrainingAnalysis({
    "Errors": PerformanceReporter(type="overall", partition="score", metrics=["mse", "mae"]),
    "Predictions": MatchResultReporter(type="per_fold", partition="test", target="target", tolerance=1.),
})
result = experiment.run(prepared, model=candidate, execution=policy, post_analysis=reports,
    refit_policy=RefitPolicy(train_positions=np.arange(len(data.X))))
cached = experiment.run(prepared, model=candidate, execution=policy, post_analysis=reports,
    refit_policy=RefitPolicy(train_positions=np.arange(len(data.X))))
assert cached.reused and cached.refit.model is None
assert result.refit.execution["worker_pid"] == os.getpid()
assert all(fold.model is None for fold in cached.training.folds)
result.to_html(output / "report.html")
print(result.record["run_id"], "reused:", cached.reused)

result.to_notebook(height=850)


aabc3873-4b6f-43c0-a7df-1634d5e550ff reused: True


## Device choice and limits

For a capable native adapter, DeviceAdapter resolves cpu/auto/cuda/cuda:N and calls
its explicit configuration hook. CUDA/auto worker counts are capped by gpu_jobs,
default 1. A CPU-only sklearn model remains CPU. The adapter owns moving model
parameters and converting input tensors; feature DataFrames are not moved by the
policy. The guide includes a complete tested Torch helper and optional example.

Real CUDA was checked separately on one installed device with a tiny synthetic
adapter. This notebook uses CPU only. Multi-GPU/distributed execution and live
Jupyter display remain unverified. Fifteen earlier notebooks are unchanged.

[Guide](../docs/analytics/execution_policy.md),
[reference](../docs/analytics/execution_policy_reference.md),
[coverage](../docs/analytics/execution_policy_documentation_checklist.md),
[verification](../docs/analytics/execution_policy_check.json).

## Known notebook shutdown limitation

On the verified Windows environment (Python 3.14.0, joblib 1.5.2), notebook cells
complete correctly, but kernel shutdown emits loky resource_tracker KeyError
tracebacks for temporary joblib_memmapping_folder paths. A one-cell plain-joblib
notebook without xDiyo imports reproduces this. max_nbytes=None also reproduces
it, so disabling automatic memmapping did not resolve the symptom.

The captured notebook run has four cleanup tracebacks; each plain-joblib control
has two. All return exit status 0 with correct numerical results. This evidence
does not establish clean notebook shutdown. Standalone guide, test and isolated
wheel computations pass. The library and installed packages were left unchanged;
no workaround or dependency upgrade is claimed. See the runtime and cleanup
control logs linked from the verification record.
